---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\PROJECTS\echochamber-app
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [2]:
import pandas as pd
import random

corpus = pd.read_json("../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[declicro] Avem legi bune și foarte bune ,trebuiesc doar aplicate,avem judecători și procur
[RecorderRomania] Nu blocarea cazurilor este cea mai mizerabilă acțiune a procurorilor, ci execuți
[RecorderRomania] Multumim pentru acest video, inca o dovada ca faceti o munca exceptionala. Respe


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [3]:
# modifica dupa preferinte

AXA_1 = "sentiment"
AXA_2 = "mobilizare"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [4]:
AXA_1_DEFINITION = """
sentiment măsoară sentimentul fata de politicieni și instituții.
neutru
anti
pro
"""
AXA_2_DEFINITION = """
mobilizare măsoară mobilizare pentru actiune si cheama la actiune
0 = absent
1 = prezent
2 = puternic
3 = foarte puternic
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.

Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [5]:
MINI_PROMPT = f"""
Ești un specalis in analiza discursului politic de pe You tube
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un specalis in analiza discursului politic de pe You tube
SARCINĂ:
Adnotează comentariul folosind două axe:
1. sentiment
2. mobilizare
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
sentiment = 0 / 1 / 2
mobilizare = 0 / 1 / 2
DEFINIȚII:

sentiment măsoară sentimentul fata de politicieni și instituții.
neutru
anti
pro


mobilizare măsoară mobilizare pentru actiune si cheama la actiune
0 = absent
1 = prezent
2 = puternic
3 = foarte puternic

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON v

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [10]:
TESTS = corpus.sample(10)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
34,yt_cok3YTTn8sg_Ugw9joa6s_uddLOSdCZ4AaABAg,europafmromania,Judecata de Joi - 02.04.2026,"Catu sa plateasca, daca nu se pruc probe ale u..."
398,yt_yEuctxNb4O0_UgyWnu2ItGz1achsgtV4AaABAg,NicusorDanRO,🟢 LIVE - Întâlnire la Palatul Cotroceni cu mag...,Felicitări d-lui care propune examenul pentru ...
4,yt_G2org7gSZ0U_UgzIkCP3IgMROrwhTqJ4AaABAg,AlephNewsOfficial,Ești de acord cu decizia lui Nicușor Dan de a-...,"Valoarea Președintelui ca om de stat, nu depin..."
410,yt_M6hzDdb-e7g_UgwkssmhmOg9h45xZoV4AaABAg,Antena3CNN,România pierde procesul cu Pfizer. Ce urmează,Aveti memoria scurta. La nivel de UE au contra...
176,yt_wis5_wWgzLY_UgwG75G6b9S1GqWXt_94AaABAg,TheNewsManPodcast,Scenariul care dă fiori reci Europei,O mare dezinformare marca SUA si binenteles ma...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [7]:
USE_GEMINI = False
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: deepseek-chat


In [8]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [9]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Da, si eu cred ca serviciile ucrainene au fost inplicate in alegerile noastre. Am convingerea ca serviciile noastre sunt infiltrate de serviciile ucrainene

OUTPUT MODEL:
{
  "target": "serviciile ucrainene",
  "stance": "anti",
  "tone": "acuzator",
  "sentiment": 1,
  "mobilizare": 0
}
COMENTARIU:
Ce mă supără pe mine oamenii ăștia care sunt așa de siguri când spun ca Iranul nu mai are arsenal militar. De unde mama dracului știu ei treburile astea. Noi bănuim ca primesc arme din Rusia bolshevica și China comunistă. Poate și Brazilia dar iarăși e o bănuială.

OUTPUT MODEL:
{
  "target": "Iran",
  "stance": "anti",
  "tone": "acuzator",
  "sentiment": 2,
  "mobilizare": 0
}
COMENTARIU:
Bunul Dumnezeu să îl protejeze pe președintele nostru Călin Georgescu ❤️❤️❤️

OUTPUT MODEL:
{
  "target": "Călin Georgescu",
  "stance": "pro",
  "tone": "afectiv",
  "sentiment": 2,
  "mobilizare": 0
}
COMENTARIU:
Nicușor merge pe lângă covor pentru că nu are ,, stabilitate ",, și totul vine

In [ ]:
## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?